# 질문셋들 테스트

할 수 있는 것:
- 질문 유형별 테스트셋 확인
- 전체 질문 일괄 실행
- 유형별 선택 실행
- 실제 Generation 호출 또는 프롬프트 미리보기(prompt only) 실행
- 결과를 `csv`/`json`으로 저장
- 응답 형식 / 출처 / 추정 여부 등을 수동 체크

권장 사용 순서:
1. 경로/모드 설정
2. 질문셋 확인
3. `RUN_MODE`를 `prompt_only` 또는 `generate`로 선택
4. 전체 또는 유형별 실행
5. 결과 확인 및 CSV 저장


In [33]:
from pathlib import Path
import sys
import os

cwd = Path.cwd().resolve()

candidate_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
]

project_root = None
for root in candidate_roots:
    if (root / "src").exists():
        project_root = root
        break

if project_root is None:
    project_root = cwd

src_path = project_root / "src"

# 프로젝트 루트와 src 폴더경로 추가
for path in [str(src_path), str(project_root)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("project_root =", project_root)
print("src_path =", src_path)
print("src_exists =", src_path.exists())


project_root = C:\develop_project\codeit-4team-2nd_project-Bidcoin\Bidcoin
src_path = C:\develop_project\codeit-4team-2nd_project-Bidcoin\Bidcoin\src
src_exists = True


In [34]:

import json
from copy import deepcopy
from datetime import datetime

try:
    import pandas as pd
except ImportError:
    pd = None

from generation.mock_data import get_mock_retrieval_result
from generation.context_builder import build_context_block, build_history_block
from generation.prompts import SYSTEM_PROMPT, build_user_prompt
from generation.generator import BidCoinGenerator
from generation.schemas import RetrievalResult
from generation.config import Settings
from rag_api import get_rag_context
from rag_api_v2 import get_rag_context2


## 1. 질문셋 정의

예상 질문 흐름을 바탕으로 만든 1차 Generation 테스트셋입니다.

각 카테고리 역할
- fact: 예산, 기관, 일정 같은 사실 추출
- condition: 가능 여부, 요구 여부, 자격 여부
- summary: 정리/요약
- compare: 비교
- recommend: 추천/판단
- evidence: 근거/원문 강조
- followup: 이전 맥락 이어받기
- refusal: 문서에 없는 질문 잘 거절하는지
- complex: 여러 요청이 섞인 난도 높은 질문


In [35]:
TEST_SETS = {
    "fact": [
        "국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.",
        "고려대학교 차세대 포털·학사 정보시스템 구축사업의 발주기관과 사업 금액을 알려줘.",
        "광주과학기술원의 학사시스템 기능개선 사업의 공개 일자와 사업 금액을 알려줘.",
        "한국원자력연구원의 한국원자력연구원 선량평가시스템 고도화 사업의 사업 금액을 알려줘.",
        "한국가스공사의 [재공고]차세대 통합정보시스템(ERP) 구축 사업의 사업 금액을 알려줘.",
        "국립중앙의료원의 2024년도 차세대 응급의료 상황관리시스템 구축 위탁용역의 사업 금액을 알려줘.",
        "한국산업단지공단의 산단 안전정보시스템 1차 구축 용역의 사업 금액과 공개 일자를 알려줘.",
        "나노종합기술원의 스마트 팹 서비스 활용체계 구축관련 설비온라인 시스템 구축 용역의 사업 금액을 알려줘."
    ],

    "summary": [
        "국민연금공단의 2024년 이러닝시스템 운영 용역의 목적과 주요 업무를 5줄 이내로 요약해줘.",
        "고려대학교 차세대 포털·학사 정보시스템 구축사업의 추진 배경과 핵심 구축 목표를 요약해줘.",
        "광주과학기술원의 학사시스템 기능개선 사업의 핵심 기능 요구사항을 요약해줘.",
        "한국연구재단의 2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선 사업의 주요 개선 내용을 요약해줘.",
        "재단법인스포츠윤리센터의 스포츠윤리센터 LMS(학습지원시스템) 기능개선 사업의 목적과 기대효과를 요약해줘.",
        "서민금융진흥원의 서민금융 채팅 상담시스템 구축 사업의 주요 요구사항을 요약해줘.",
        "서울특별시 여성가족재단의 서울 디지털성범죄 안심지원센터 업무시스템 및 AI 기반 삭제지원시스템 통합 사업의 핵심 내용을 요약해줘.",
        "대한적십자사 의료원의 적십자병원 병원정보 재해복구시스템 구축 용역의 구축 목적과 범위를 요약해줘."
    ],

    "condition": [
        "국민연금공단의 2024년 이러닝시스템 운영 용역에 유지보수 또는 운영지원 요구사항이 있는지 알려줘.",
        "고려대학교 차세대 포털·학사 정보시스템 구축사업에 응답시간이나 성능 관련 요구사항이 있는지 알려줘.",
        "광주과학기술원의 학사시스템 기능개선 사업에 보안 또는 개인정보보호 관련 요구사항이 있는지 알려줘.",
        "한국연구재단의 2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선 사업에 클라우드 운영 관련 요구사항이 있는지 알려줘.",
        "한국원자력연구원의 선량평가시스템 고도화 사업에 보안 요구사항이 있는지 알려줘.",
        "국립중앙의료원의 차세대 응급의료 상황관리시스템 구축 위탁용역에 재해복구 또는 이중화 관련 요구사항이 있는지 알려줘.",
        "한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업에 대기업 참여 가능 여부나 참가 자격 관련 조건이 있는지 알려줘.",
        "국가철도공단의 철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역에 산출물 또는 보고서 제출 요구가 있는지 알려줘."
    ],

    "evidence": [
        "국민연금공단의 2024년 이러닝시스템 운영 용역에서 주요 업무 범위를 보여주는 근거 문장을 출처와 함께 제시해줘.",
        "고려대학교 차세대 포털·학사 정보시스템 구축사업에서 구축 목표를 보여주는 근거 문장을 제시해줘.",
        "광주과학기술원의 학사시스템 기능개선 사업에서 기능개선 범위를 보여주는 근거 문장을 제시해줘.",
        "한국연구재단의 2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선 사업에서 클라우드 서버 운영 관련 근거 문장을 제시해줘.",
        "한국원자력연구원의 선량평가시스템 고도화 사업에서 추진 목적을 보여주는 근거 문장을 제시해줘.",
        "국립중앙의료원의 차세대 응급의료 상황관리시스템 구축 위탁용역에서 응급의료 상황관리 관련 핵심 요구를 보여주는 근거 문장을 제시해줘.",
        "서울특별시 여성가족재단의 AI 기반 삭제지원시스템 통합 사업에서 AI 관련 언급이 있다면 그 근거 문장을 제시해줘.",
        "한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업에서 ERP 관련 범위를 보여주는 근거 문장을 제시해줘."
    ],

    "compare": [
        "고려대학교 차세대 포털·학사 정보시스템 구축사업과 광주과학기술원 학사시스템 기능개선 사업의 목적과 범위를 비교해줘.",
        "국민연금공단의 2024년 이러닝시스템 운영 용역과 재단법인스포츠윤리센터의 스포츠윤리센터 LMS(학습지원시스템) 기능개선 사업의 성격 차이를 비교해줘.",
        "한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업과 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업(재공고)의 사업 규모와 성격을 비교해줘.",
        "경상북도 봉화군의 봉화군 재난통합관리시스템 고도화 사업과 한국산업단지공단의 산단 안전정보시스템 1차 구축 용역의 안전 관련 시스템 구축 목적을 비교해줘.",
        "국립중앙의료원의 차세대 응급의료 상황관리시스템 구축 사업과 대한적십자사 의료원의 병원정보 재해복구시스템 구축 용역의 의료 분야 시스템 성격 차이를 비교해줘.",
        "한국연구재단의 대학산학협력활동 실태조사 시스템(UICC) 기능개선 사업과 2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구축 사업의 업무 범위를 비교해줘.",
        "한국수자원공사의 건설통합시스템(CMS) 고도화와 수도사업장 통합 사고분석솔루션 시범구축 용역의 사업 목적을 비교해줘.",
        "한국생산기술연구원의 EIP3.0 고압가스 안전관리 시스템 구축 용역과 한국전기안전공사의 전기안전 관제시스템 보안 모듈 개발 용역의 안전·보안 초점을 비교해줘."
    ],

    "complex": [
        "국민연금공단의 2024년 이러닝시스템 운영 용역에 대해 목적, 주요 업무, 운영 범위, 유지보수 요구, 일정 정보를 항목별로 정리해줘.",
        "고려대학교 차세대 포털·학사 정보시스템 구축사업에 대해 추진 배경, 구축 목표, 주요 기능, 일정, 제출 관련 정보를 표처럼 구조화해서 정리해줘.",
        "광주과학기술원의 학사시스템 기능개선 사업이 대학 학사시스템 구축 경험이 있는 업체에게 적합한지 문서 근거만 바탕으로 제한적으로 판단해줘.",
        "한국연구재단의 2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선 사업에 대해 기능개선, 운영지원, 데이터 관리, 연계 요구사항을 구분해서 정리해줘.",
        "국립중앙의료원의 차세대 응급의료 상황관리시스템 구축 위탁용역에 대해 기능 요구사항, 비기능 요구사항, 보안 관련 요구사항을 나눠서 정리해줘.",
        "한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업에 대해 예산 규모, 사업 성격, 복잡도, 참여 난이도를 문서 근거 중심으로 정리해줘.",
        "국가철도공단의 철도인프라 디지털트윈 정보화전략계획(ISP) 수립 용역에서 최종 산출물과 수행 범위를 내부 검토 메모 형식으로 정리해줘.",
        "서울특별시 여성가족재단의 AI 기반 삭제지원시스템 통합 사업에 대해 업무시스템과 AI 시스템 통합 관점에서 핵심 확인 포인트를 체크리스트로 정리해줘."
    ],

    "refusal": [
        "삼성전자가 발주한 RFP가 이 데이터셋에 있는지 확인해줘.",
        "국민연금공단의 2024년 이러닝시스템 운영 용역에서 AWS 사용이 필수인지 알려줘. 문서에 없으면 없다고 말해줘.",
        "고려대학교 차세대 포털·학사 정보시스템 구축사업의 실제 수주사를 알려줘.",
        "한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업에서 어떤 업체가 가장 유리할지 추측해서 말해줘.",
        "광주과학기술원의 학사시스템 기능개선 사업에서 문서에 없는 세부 기술 스택까지 추정해서 설명해줘."
    ],

    "fuzzy_org": [
        "국민연금 이러닝 시스템 운영 사업의 주요 업무를 정리해줘.",
        "고려대 포털 학사 시스템 구축 사업의 구축 목표를 알려줘.",
        "광주과기원 학사 시스템 기능개선 사업의 주요 기능을 알려줘.",
        "원자력연구원 선량평가 시스템 고도화 사업의 목적을 알려줘.",
        "가스공사 차세대 ERP 구축 사업의 예산 규모를 알려줘.",
        "봉화군 재난통합관리 시스템 고도화 사업의 핵심 요구사항을 알려줘."
    ],

    "budget_and_date": [
        "국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 마감일을 함께 알려줘.",
        "한국연구재단의 2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선 사업의 공개 일자, 입찰 시작일, 입찰 마감일을 알려줘.",
        "고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업 금액이 광주과학기술원 학사시스템 기능개선 사업보다 얼마나 큰지 비교해줘.",
        "국립중앙의료원의 차세대 응급의료 상황관리시스템 구축 위탁용역과 인천공항운영서비스㈜ 차세대 ERP시스템 구축 사업의 사업 금액을 비교해줘.",
        "국가철도공단의 철도인프라 디지털트윈 ISP 수립 용역과 서울특별시교육청 지능정보화전략계획(ISP) 수립 사업의 사업 금액을 비교해줘."
    ]
}

In [36]:

def flatten_test_sets(test_sets: dict) -> list[dict]:
    rows = []
    for category, questions in test_sets.items():
        for i, q in enumerate(questions, start=1):
            rows.append({
                "category": category,
                "question_no": i,
                "question": q,
            })
    return rows

test_rows = flatten_test_sets(TEST_SETS)
print(f"총 질문 수: {len(test_rows)}")

if pd is not None:
    test_df = pd.DataFrame(test_rows)
    display(test_df)
else:
    test_rows[:5]


총 질문 수: 64


,category,question_no,question
0,fact,1,국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 ...
1,fact,2,고려대학교 차세대 포털·학사 정보시스템 구축사업의 발주기관과 사업 금액을 알려줘.
2,fact,3,광주과학기술원의 학사시스템 기능개선 사업의 공개 일자와 사업 금액을 알려줘.
3,fact,4,한국원자력연구원의 한국원자력연구원 선량평가시스템 고도화 사업의 사업 금액을 알려줘.
4,fact,5,한국가스공사의 [재공고]차세대 통합정보시스템(ERP) 구축 사업의 사업 금액을 알려줘.
...,...,...,...
59,budget_and_date,1,국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 마감일을 함께 ...
60,budget_and_date,2,한국연구재단의 2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선 사업의...
61,budget_and_date,3,고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업 금액이 광주과학기술원 학사시...
62,budget_and_date,4,국립중앙의료원의 차세대 응급의료 상황관리시스템 구축 위탁용역과 인천공항운영서비스㈜ ...


## 2. 실행 모드 설정

### `RUN_MODE`
- `prompt_only`: OpenAI 호출 없이 **프롬프트만 미리보기**
- `generate`: 실제 모델 호출

### `INPUT_MODE`
- `mock`: `mock_data.py`의 RetrievalResult 사용
- `json`: 특정 json 파일을 불러와 사용
- `retrieval_function`: **질문마다 `get_rag_context2(question, [])`를 직접 호출**해서 Retrieval 결과 사용

> 현재 권장: `generate + retrieval_function`


In [37]:
# ===== 사용자 설정 =====
RUN_MODE = "generate"   # "prompt_only", "generate" 중에서 선택
INPUT_MODE = "retrieval_function"   # "mock", "json", "retrieval_function" 중에서 선택
USE_HISTORY = False

# INPUT_MODE == "json" 일 때만 사용
RETRIEVAL_JSON_PATH = ""

# 실행할 유형 선택
# 예: ["fact", "summary"] / None 이면 전체 실행
SELECTED_CATEGORIES = None

# 각 유형에서 몇 개씩만 돌릴지 제한하고 싶다면 숫자 지정
# None 이면 해당 유형 전체
MAX_PER_CATEGORY = 2

# 실제 generation 호출 시 결과를 너무 길게 출력하지 않기 위한 preview 길이
ANSWER_PREVIEW_CHARS = 500


## 3. Retrieval 입력 로드 유틸리티

In [38]:
def load_retrieval_result(question: str, input_mode: str = "mock", json_path: str = ""):
    if input_mode == "mock":
        rr = get_mock_retrieval_result()
        rr.question = question
        return rr

    if input_mode == "json":
        if not json_path:
            raise ValueError("INPUT_MODE='json'이면 RETRIEVAL_JSON_PATH를 지정해야 합니다.")

        path = Path(json_path)
        if not path.exists():
            raise FileNotFoundError(f"JSON 파일을 찾을 수 없습니다: {path}")

        data = json.loads(path.read_text(encoding="utf-8"))
        data["question"] = question
        return RetrievalResult.model_validate(data)

    if input_mode == "retrieval_function":
        # 현재는 히스토리 없이 Retrieval 호출
        chat_history = []
        raw_result = get_rag_context2(question, chat_history)
        return RetrievalResult.model_validate(raw_result)

    raise ValueError("input_mode는 'mock', 'json', 'retrieval_function' 중 하나여야 합니다.")


## 4. 실제 모델 호출

질문 1개에 대해 실제 모델을 호출하여 결과를 확인합니다.

In [39]:
preview_question = TEST_SETS["fact"][0]
preview_result = load_retrieval_result(
    question=preview_question,
    input_mode=INPUT_MODE,
    json_path=RETRIEVAL_JSON_PATH,
)

print("preview_question =", preview_question)
print("contexts 개수 =", len(preview_result.contexts))
preview_result



[RAG API 가동] 사용자 질의 접수: '국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.'
[Step 2: 질문 재구성] 국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.
[Step 3: HyDE 가상 문서 생성] 검색 명중률 증폭 완료
임베딩 모델 로드: text-embedding-3-small
[Step 4: 검색 & 랭킹] 최상위 문서 추출 및 유사도 계산 완료
[Step 5: 문맥 압축 가동]
    [압축 성공] 국민연금공단_2024년 이러닝시스템 운영 용역.hwp (1235자 ➡️ 164자)
   [내용 없음 버림] 국민연금공단_2024년 이러닝시스템 운영 용역.hwp
    [압축 성공] 국민연금공단_2024년 이러닝시스템 운영 용역.hwp (918자 ➡️ 16자)
[Step 6: 자가 반성] 최종 팩트 체크 진행 중...
    [Self-RAG 경고] 문서 내용은 있으나 질문에 완벽히 답하기 부족함. 환각 방지를 위해 결과 초기화.
 [RAG API 완료] 최종 유효 문서 0개 반환
preview_question = 국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.
contexts 개수 = 0


RetrievalResult(question='국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.', contexts=[], chat_history=[])

## 5. Prompt 확인

샘플 질문 1개에 대해 모델에 어떤 문자열이 들어가는지 확인합니다.
`preview_result`는 현재 `INPUT_MODE`에 맞춰 생성됩니다.


In [40]:
def make_prompt_preview(retrieval_result):
    settings = Settings()
    context_block, used_sources = build_context_block(
        contexts=retrieval_result.contexts,
        max_contexts=settings.max_contexts,
        max_chars=settings.max_context_chars,
    )
    history_block = build_history_block(
        retrieval_result.chat_history
    )
    user_prompt = build_user_prompt(
        question=retrieval_result.question,
        context_block=context_block,
        history_block=history_block,
    )
    return {
        "context_block": context_block,
        "history_block": history_block,
        "user_prompt": user_prompt,
        "used_sources": used_sources,
    }

preview = make_prompt_preview(preview_result)
print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT)
print("\n=== USER PROMPT ===")
print(preview["user_prompt"])
print("\n=== USED SOURCES ===")
print(preview["used_sources"])


=== SYSTEM PROMPT ===
당신은 B2G 공공입찰 전문 컨설팅 어시스턴트입니다.
반드시 주어진 RFP 문서 컨텍스트만을 근거로만 답변하세요.

공통 규칙:
1. 문서에 없는 내용은 추측하지 말고 "해당 문서에서 확인할 수 없습니다."라고 답하세요.
2. 답변은 한국어로 작성하세요.
3. 문서 근거가 있는 내용만 답하세요.
4. 마지막에 반드시 "출처" 섹션을 넣고 파일명을 나열하세요.
5. 사용자가 판단/추천을 요청하더라도 문서에 드러난 근거 범위 안에서만 제한적으로 설명하세요.
6. 불확실한 내용은 "불확실"로 표시하세요.

=== USER PROMPT ===
[참고 문서]
검색된 문서가 없습니다.

[최근 대화]
없음

[질문]
국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.

[답변 형식 선택 규칙]
- 질문 의도를 먼저 파악한 뒤 가장 적절한 형식을 선택하세요.
- 하나의 질문이 여러 유형을 동시에 포함하면, 더 핵심적인 요청을 우선하세요.
- 비교 요청이 있으면 비교 형식을 우선하세요.
- 추천/판단 요청이 있으면 추천 근거와 리스크를 함께 제시하세요.
- 근거/원문 요청이 있으면 근거 문장을 별도 항목으로 제시하세요.
- 후속 질문이면 최근 대화 맥락을 반영하세요.
- 문서 범위를 벗어나면 "해당 문서에서 확인할 수 없습니다."라고 답하세요.

[사실 확인형]
예: 예산, 기관, 일정, 제출 방식, 제출 서류
형식:
1. 답변
2. 근거
3. 출처

[조건 확인형]
예: 자격요건, 유지보수 포함 여부, 공동수급 가능 여부, 보안 요구 존재 여부
형식:
1. 확인 결과 (있음 / 없음 / 불확실)
2. 세부 내용
3. 근거
4. 출처

[요약형]
예: 목적, 배경, 핵심 요구사항, 기능/비기능 정리
형식:
1. 요약
2. 상세 정리
3. 핵심 근거
4. 출처

[비교형]
예: 두 사업 비교, 공통점/차이점, 어느 쪽이 더 어려운지
형식:
1. 비교 요약
2. 공통점
3. 차이점
4. 판단 포인트


## 6. 단일 질문 실행 함수

이 함수는 질문 1개를 실행합니다.
- `prompt_only` 모드면 프롬프트만 생성
- `generate` 모드면 실제 모델 호출
- **중요: 질문마다 Retrieval을 다시 호출**합니다.


In [41]:
def run_single_question(question: str, run_mode: str = "prompt_only") -> dict:
    rr = load_retrieval_result(
        question=question,
        input_mode=INPUT_MODE,
        json_path=RETRIEVAL_JSON_PATH,
    )

    prompt_preview = make_prompt_preview(rr)

    result = {
        "question": question,
        "used_sources": prompt_preview["used_sources"],
        "status": "ok",
        "mode": run_mode,
        "answer": None,
        "answer_preview": None,
        "context_preview": prompt_preview["context_block"][:800],
        "user_prompt_preview": prompt_preview["user_prompt"][:2000],
        "error": None,
    }

    if run_mode == "prompt_only":
        result["answer"] = "[PROMPT ONLY MODE] 실제 모델 호출 없이 프롬프트만 생성했습니다."
        result["answer_preview"] = result["answer"]
        return result

    if run_mode == "generate":
        try:
            generator = BidCoinGenerator()
            response = generator.generate(rr)
            result["answer"] = response.answer
            result["answer_preview"] = response.answer[:ANSWER_PREVIEW_CHARS]
            result["used_context_count"] = response.used_context_count
            result["raw_model_output"] = response.raw_model_output
            return result
        except Exception as e:
            result["status"] = "error"
            result["error"] = str(e)
            return result

    raise ValueError("run_mode는 'prompt_only' 또는 'generate' 이어야 합니다.")


## 7. 단일 질문 수동 테스트

In [42]:
sample_question = TEST_SETS["fact"][0]  # 필요하면 바꿔도 됨
single_result = run_single_question(sample_question, RUN_MODE)

print("=== QUESTION ===")
print(single_result["question"])
print("\n=== STATUS ===")
print(single_result["status"])
print("\n=== ANSWER PREVIEW ===")
print(single_result["answer_preview"])
print("\n=== USED SOURCES ===")
print(single_result["used_sources"])
print("\n=== USER PROMPT PREVIEW ===")
print(single_result["user_prompt_preview"])



[RAG API 가동] 사용자 질의 접수: '국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.'
[Step 2: 질문 재구성] 국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.
[Step 3: HyDE 가상 문서 생성] 검색 명중률 증폭 완료
임베딩 모델 로드: text-embedding-3-small
[Step 4: 검색 & 랭킹] 최상위 문서 추출 및 유사도 계산 완료
[Step 5: 문맥 압축 가동]

------------------------------------------------------------
 [Debug] 문서 필터링 됨 (커트라인 미달)
 출처: 한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp
 점수: 0.03 (기준: 0.8)
 미리보기: [발주기관: 한국연구재단 | 사업명: 2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구축 사업]  ❍ 협상 대상자가 제안한 내용을 가감하는 경우 가감내용에 상당하는 금액...

------------------------------------------------------------

------------------------------------------------------------
 [Debug] 문서 필터링 됨 (커트라인 미달)
 출처: 한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp
 점수: 0.01 (기준: 0.8)
 미리보기: [발주기관: 한국철도공사 (용역) | 사업명: 예약발매시스템 개량 ISMP 용역]  Fax : ( 02) 361 – 8269 Email : hyungsub@korail.com 마....

------------------------------------------------------------

------------------------------------------------------------
 [Debu

## 8. 배치 실행 함수

- 전체 질문 일괄 실행
- 유형별 선택 실행
- 유형별 개수 제한 실행
- 각 질문마다 Retrieval을 새로 호출


In [43]:
def select_questions(test_sets, selected_categories=None, max_per_category=None):
    selected = []
    categories = selected_categories or list(test_sets.keys())

    for category in categories:
        questions = test_sets[category]
        if max_per_category is not None:
            questions = questions[:max_per_category]

        for i, q in enumerate(questions, start=1):
            selected.append({
                "category": category,
                "question_no": i,
                "question": q,
            })
    return selected


def run_batch(test_sets, run_mode="prompt_only",
              selected_categories=None, max_per_category=None):
    selected_rows = select_questions(
        test_sets=test_sets,
        selected_categories=selected_categories,
        max_per_category=max_per_category,
    )

    outputs = []
    total = len(selected_rows)

    for idx, row in enumerate(selected_rows, start=1):
        print(f"[{idx}/{total}] {row['category']} - {row['question_no']}: {row['question']}")
        out = run_single_question(
            question=row["question"],
            run_mode=run_mode,
        )
        out["category"] = row["category"]
        out["question_no"] = row["question_no"]
        outputs.append(out)

    return outputs


## 9. 전체 또는 유형별 일괄 실행

이 셀을 실행하면 질문셋을 한 번에 돌립니다.
- `RUN_MODE='prompt_only'`면 빠르게 전체 프롬프트 구조 점검
- `RUN_MODE='generate'`면 실제 답변 생성
- `INPUT_MODE='retrieval_function'`이면 **질문마다 `get_rag_context2()`를 다시 호출**합니다.


In [44]:
batch_results = run_batch(
    test_sets=TEST_SETS,
    run_mode=RUN_MODE,
    selected_categories=SELECTED_CATEGORIES,
    max_per_category=MAX_PER_CATEGORY,
)

print(f"총 실행 결과 수: {len(batch_results)}")


[1/18] fact - 1: 국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.

[RAG API 가동] 사용자 질의 접수: '국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.'
[Step 2: 질문 재구성] 국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 알려줘.
[Step 3: HyDE 가상 문서 생성] 검색 명중률 증폭 완료
임베딩 모델 로드: text-embedding-3-small
[Step 4: 검색 & 랭킹] 최상위 문서 추출 및 유사도 계산 완료
[Step 5: 문맥 압축 가동]
    [압축 성공] 국민연금공단_2024년 이러닝시스템 운영 용역.hwp (1235자 ➡️ 248자)
   [내용 없음 버림] 국민연금공단_2024년 이러닝시스템 운영 용역.hwp
   [내용 없음 버림] 국민연금공단_2024년 이러닝시스템 운영 용역.hwp
[Step 6: 자가 반성] 최종 팩트 체크 진행 중...
    [Self-RAG 경고] 문서 내용은 있으나 질문에 완벽히 답하기 부족함. 환각 방지를 위해 결과 초기화.
 [RAG API 완료] 최종 유효 문서 0개 반환
[2/18] fact - 2: 고려대학교 차세대 포털·학사 정보시스템 구축사업의 발주기관과 사업 금액을 알려줘.

[RAG API 가동] 사용자 질의 접수: '고려대학교 차세대 포털·학사 정보시스템 구축사업의 발주기관과 사업 금액을 알려줘.'
[Step 2: 질문 재구성] 고려대학교 차세대 포털·학사 정보시스템 구축사업의 발주기관과 사업 금액을 알려줘.
[Step 3: HyDE 가상 문서 생성] 검색 명중률 증폭 완료
임베딩 모델 로드: text-embedding-3-small
[Step 4: 검색 & 랭킹] 최상위 문서 추출 및 유사도 계산 완료
[Step 5: 문맥 압축 가동]
    [압축 성공] 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf (1010자 ➡️ 88

In [45]:

if pd is not None:
    results_df = pd.DataFrame(batch_results)
    display(
        results_df[
            [
                "category",
                "question_no",
                "question",
                "status",
                "answer_preview",
                "used_sources",
                "error",
            ]
        ]
    )
else:
    batch_results[:3]


,category,question_no,question,status,answer_preview,used_sources,error
0,fact,1,국민연금공단의 2024년 이러닝시스템 운영 용역의 사업 금액과 입찰 참여 마감일을 ...,ok,"1. 답변\n해당 문서에서 확인할 수 없습니다. (사업 금액: 불확실, 입찰 참여 ...",[],None
1,fact,2,고려대학교 차세대 포털·학사 정보시스템 구축사업의 발주기관과 사업 금액을 알려줘.,ok,"1. 답변\n- 발주기관: 고려대학교\n- 사업 금액: 11,270,000,000원...",[고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf],None
2,summary,1,국민연금공단의 2024년 이러닝시스템 운영 용역의 목적과 주요 업무를 5줄 이내로 ...,ok,1. 요약\n국민연금공단의 2024년 이러닝시스템 운영 용역 목적은 모바일 포함 안...,[국민연금공단_2024년 이러닝시스템 운영 용역.hwp],None
3,summary,2,고려대학교 차세대 포털·학사 정보시스템 구축사업의 추진 배경과 핵심 구축 목표를 요...,ok,"1. 요약\n- 추진 배경: 노후화된 검색엔진, 모바일 서비스 미흡, 양 캠퍼스(서...",[고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf],None
4,condition,1,국민연금공단의 2024년 이러닝시스템 운영 용역에 유지보수 또는 운영지원 요구사항이...,ok,1. 확인 결과\n- 있음\n\n2. 세부 내용\n- 이러닝 시스템의 구축 및 운영...,[국민연금공단_2024년 이러닝시스템 운영 용역.hwp],None
5,condition,2,고려대학교 차세대 포털·학사 정보시스템 구축사업에 응답시간이나 성능 관련 요구사항이...,ok,1. 확인 결과\n해당 문서에서 확인할 수 없습니다.\n\n2. 확인할 수 없는 이...,[한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp],None
6,evidence,1,국민연금공단의 2024년 이러닝시스템 운영 용역에서 주요 업무 범위를 보여주는 근거...,ok,1. 답변\n다음은 국민연금공단의 2024년 이러닝시스템 운영 용역의 주요 업무 범...,[국민연금공단_2024년 이러닝시스템 운영 용역.hwp],None
7,evidence,2,고려대학교 차세대 포털·학사 정보시스템 구축사업에서 구축 목표를 보여주는 근거 문장...,ok,1. 답변\n다음은 고려대학교 차세대 포털·학사 정보시스템 구축사업의 구축 목표를 ...,[고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf],None
8,compare,1,고려대학교 차세대 포털·학사 정보시스템 구축사업과 광주과학기술원 학사시스템 기능개선...,ok,1. 비교 요약\n해당 문서들이 제공되지 않아 고려대학교 차세대 포털·학사 정보시스...,[],None
9,compare,2,국민연금공단의 2024년 이러닝시스템 운영 용역과 재단법인스포츠윤리센터의 스포츠윤리...,ok,1. 비교 요약\n해당 문서에서 확인할 수 없습니다.\n\n2. 공통점\n해당 문서...,[],None


## 10. 결과 저장

- batch 결과: 실행 결과 전체
- manual eval: 사람이 체크한 평가표


In [ ]:

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = project_root / "outputs"
output_dir.mkdir(exist_ok=True)

batch_json_path = output_dir / f"generation_batch_results_{timestamp}.json"
batch_json_path.write_text(
    json.dumps(batch_results, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("saved:", batch_json_path)

if pd is not None:
    batch_csv_path = output_dir / f"generation_batch_results_{timestamp}.csv"
    results_df.to_csv(batch_csv_path, index=False, encoding="utf-8-sig")
    print("saved:", batch_csv_path)


saved: C:\develop_project\codeit-4team-2nd_project-Bidcoin\Bidcoin\outputs\generation_batch_results_20260413_181443.json
saved: C:\develop_project\codeit-4team-2nd_project-Bidcoin\Bidcoin\outputs\generation_batch_results_20260413_181443.csv


: 

## 11. 사용 시나리오

### A. 프롬프트만 빠르게 확인하고 싶을 때
- `RUN_MODE = "prompt_only"`
- `INPUT_MODE = "mock"` 또는 `"retrieval_function"`

### B. 실제 Retrieval 함수로 end-to-end 테스트할 때
- `RUN_MODE = "generate"`
- `INPUT_MODE = "retrieval_function"`
- 질문셋의 **각 질문마다 `get_rag_context2(question, [])` 호출**

### C. 유형별로만 돌리고 싶을 때
예:
```python
SELECTED_CATEGORIES = ["fact", "condition", "summary"]
```

### D. 각 유형당 2개씩만 먼저 보고 싶을 때
```python
MAX_PER_CATEGORY = 2
```
